# NetShield — Phase 11: Leakage-Safe Threshold Transfer & Calibration

**Research question:** Does the strong CORAL ranking improvement become useful cross-dataset detection once the decision threshold is selected without using CICIDS evaluation labels?

**Leakage rule:** CICIDS evaluation labels are used only after the threshold and model are frozen. The final threshold is selected from an UNSW source-validation split.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
warnings.filterwarnings("ignore")
ROOT=Path.cwd(); ROOT=ROOT.parent if ROOT.name.lower()=="notebooks" else ROOT
RAW_UNSW=ROOT/'data/raw/UNSW-NB15'; RAW_CICIDS=ROOT/'data/raw/CICIDS2017'
OUT=ROOT/'outputs/threshold_transfer_analysis'; TABLE=OUT/'tables'; PLOTS=OUT/'plots'; TABLE.mkdir(parents=True,exist_ok=True); PLOTS.mkdir(parents=True,exist_ok=True)
SEED=42
print(ROOT)

## 1. Load official UNSW-NB15 train/test

In [ ]:
def find_file(folder,names):
    wanted={x.lower() for x in names}
    return next((p for p in folder.rglob('*') if p.is_file() and p.name.lower() in wanted),None)
train_p=find_file(RAW_UNSW,['UNSW_NB15_training-set.csv','UNSW-NB15_training-set.csv'])
test_p=find_file(RAW_UNSW,['UNSW_NB15_testing-set.csv','UNSW-NB15_testing-set.csv'])
if train_p is None or test_p is None: raise FileNotFoundError('Official UNSW train/test files not found.')
unsw_train=pd.read_csv(train_p); unsw_test=pd.read_csv(test_p)
print(unsw_train.shape,unsw_test.shape)

## 2. Load CICIDS fields needed for the common semantic representation

In [ ]:
NEEDED=['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets','Total Length of Bwd Packets','Flow Packets/s','Fwd Packets/s','Bwd Packets/s','Average Packet Size','Label']
frames=[]
for p in sorted(RAW_CICIDS.rglob('*.csv')):
    h=pd.read_csv(p,nrows=0)
    lookup={str(x).strip().replace('\ufeff','').lower():x for x in h.columns}
    sel={x:lookup[x.lower()] for x in NEEDED if x.lower() in lookup}
    if 'Label' not in sel: print('skip',p.name); continue
    z=pd.read_csv(p,usecols=list(sel.values()),low_memory=False).rename(columns={v:k for k,v in sel.items()})
    z['source_file']=p.name; frames.append(z); print('loaded',p.name,len(z))
if not frames: raise ValueError('No CICIDS CSV with Label found.')
cicids=pd.concat(frames,ignore_index=True); print('CICIDS',cicids.shape)

## 3. Build the 11-feature semantically aligned representation

In [ ]:
COMMON=['flow_duration_sec','fwd_packets','bwd_packets','fwd_bytes','bwd_bytes','total_packets','total_bytes','packet_rate','fwd_packet_rate','bwd_packet_rate','avg_packet_size']
def unsw_common(d):
    dur=pd.to_numeric(d.dur,errors='coerce'); fp=pd.to_numeric(d.spkts,errors='coerce'); bp=pd.to_numeric(d.dpkts,errors='coerce'); fb=pd.to_numeric(d.sbytes,errors='coerce'); bb=pd.to_numeric(d.dbytes,errors='coerce'); td=dur.replace(0,np.nan); tp=(fp+bp).replace(0,np.nan)
    return pd.DataFrame({'flow_duration_sec':dur,'fwd_packets':fp,'bwd_packets':bp,'fwd_bytes':fb,'bwd_bytes':bb,'total_packets':fp+bp,'total_bytes':fb+bb,'packet_rate':(fp+bp)/td,'fwd_packet_rate':fp/td,'bwd_packet_rate':bp/td,'avg_packet_size':(fb+bb)/tp},index=d.index)[COMMON]
def cic_common(d):
    dur=pd.to_numeric(d['Flow Duration'],errors='coerce')/1e6; fp=pd.to_numeric(d['Total Fwd Packets'],errors='coerce'); bp=pd.to_numeric(d['Total Backward Packets'],errors='coerce'); fb=pd.to_numeric(d['Total Length of Fwd Packets'],errors='coerce'); bb=pd.to_numeric(d['Total Length of Bwd Packets'],errors='coerce')
    return pd.DataFrame({'flow_duration_sec':dur,'fwd_packets':fp,'bwd_packets':bp,'fwd_bytes':fb,'bwd_bytes':bb,'total_packets':fp+bp,'total_bytes':fb+bb,'packet_rate':pd.to_numeric(d['Flow Packets/s'],errors='coerce'),'fwd_packet_rate':pd.to_numeric(d['Fwd Packets/s'],errors='coerce'),'bwd_packet_rate':pd.to_numeric(d['Bwd Packets/s'],errors='coerce'),'avg_packet_size':pd.to_numeric(d['Average Packet Size'],errors='coerce')},index=d.index)[COMMON]
X=unsw_common(unsw_train); y=pd.to_numeric(unsw_train.label).astype('int8').to_numpy(); Xte=unsw_common(unsw_test); yte=pd.to_numeric(unsw_test.label).astype('int8').to_numpy(); Xt=cic_common(cicids); yt=cicids.Label.astype(str).str.strip().str.upper().ne('BENIGN').astype('int8').to_numpy()
print(X.shape,Xt.shape)

## 4. Leakage-safe source validation and CICIDS adaptation/evaluation split

In [ ]:
def clean(X):
    X=np.asarray(X,dtype=np.float64).copy(); X[~np.isfinite(X)]=np.nan; X[np.abs(X)>np.finfo(np.float32).max]=np.nan; return X
sfit,sval=train_test_split(np.arange(len(X)),test_size=.20,stratify=y,random_state=SEED)
cpool,ceval=train_test_split(np.arange(len(Xt)),test_size=.20,stratify=yt,random_state=SEED)
rng=np.random.default_rng(SEED); adapt_idx=rng.choice(cpool,size=min(300000,len(cpool)),replace=False)
Xsf=X.iloc[sfit]; ysf=y[sfit]; Xsv=X.iloc[sval]; ysv=y[sval]; Xta=Xt.iloc[adapt_idx]; Xev=Xt.iloc[ceval]; yev=yt[ceval]
print('source fit/val',Xsf.shape,Xsv.shape,'target adapt/eval',Xta.shape,Xev.shape)

## 5. Correct CORAL transform

In [ ]:
def fit_std(X):
    imp=SimpleImputer(strategy='median'); a=imp.fit_transform(clean(X)); sc=StandardScaler(); return imp,sc,sc.fit_transform(a)
def tr_std(X,imp,sc): return sc.transform(imp.transform(clean(X)))
def cov(Z): return np.atleast_2d(np.cov(np.asarray(Z),rowvar=False))
def ssqrt(M,inv=False,eps=1e-6):
    v,Q=np.linalg.eigh((M+M.T)/2); v=np.clip(v,eps,None); v=1/np.sqrt(v) if inv else np.sqrt(v); return (Q*v)@Q.T
imp,sc,Zs=fit_std(Xsf); Zt=tr_std(Xta,imp,sc); Cs=cov(Zs); Ct=cov(Zt); A=ssqrt(Cs,True)@ssqrt(Ct)
ZsA=(Zs-Zs.mean(0))@A+Zt.mean(0)
def src_coral(X): return (tr_std(X,imp,sc)-Zs.mean(0))@A+Zt.mean(0)
def tgt_coral(X): return tr_std(X,imp,sc)
XsvA=src_coral(Xsv); XevA=tgt_coral(Xev)
cb=np.linalg.norm(Cs-Ct,'fro'); ca=np.linalg.norm(cov(ZsA)-Ct,'fro')
print('CORAL covariance distance:',cb,'->',ca)
pd.DataFrame([{'before':cb,'after':ca,'reduction':1-ca/cb}]).to_csv(TABLE/'01_coral_sanity.csv',index=False)
if not np.isfinite(ca) or ca>=cb: raise RuntimeError('CORAL sanity check failed; stop.')

## 6. Train baseline and CORAL Random Forests

In [ ]:
bimp=SimpleImputer(strategy='median'); XsfB=bimp.fit_transform(clean(Xsf)); XevB=bimp.transform(clean(Xev))
params=dict(n_estimators=250,random_state=SEED,n_jobs=-1,max_features='sqrt',min_samples_leaf=2)
rfB=RandomForestClassifier(**params).fit(XsfB,ysf); rfC=RandomForestClassifier(**params).fit(ZsA,ysf)
pB=rfB.predict_proba(XevB)[:,1]; pC=rfC.predict_proba(XevA)[:,1]; pSV=rfC.predict_proba(XsvA)[:,1]
print('models ready')

## 7. Select threshold from UNSW validation only

In [ ]:
grid=np.round(np.arange(.05,.951,.01),2); rows=[]
for t in grid:
    pr=(pSV>=t).astype('int8'); rows.append({'threshold':t,'precision':precision_score(ysv,pr,zero_division=0),'recall':recall_score(ysv,pr,zero_division=0),'f1':f1_score(ysv,pr,zero_division=0),'macro_f1':f1_score(ysv,pr,average='macro',zero_division=0)})
th=pd.DataFrame(rows); best=th.loc[th.f1.idxmax()]; T=float(best.threshold); print('FROZEN THRESHOLD =',T); th.to_csv(TABLE/'02_source_threshold_search.csv',index=False)
plt.figure(figsize=(9,5)); plt.plot(th.threshold,th.f1); plt.axvline(T,linestyle='--'); plt.xlabel('Threshold'); plt.ylabel('UNSW validation F1'); plt.title('Source-only threshold selection'); plt.tight_layout(); plt.savefig(PLOTS/'01_source_threshold_selection.png',dpi=160); plt.show()

## 8. Frozen CICIDS evaluation

In [ ]:
def metrics(y,p,t,name):
    pred=(p>=t).astype('int8'); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); return {'model':name,'threshold':t,'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),'f1':f1_score(y,pred,zero_division=0),'macro_f1':f1_score(y,pred,average='macro',zero_division=0),'pr_auc':average_precision_score(y,p),'roc_auc':roc_auc_score(y,p),'fpr':fp/(fp+tn),'fnr':fn/(fn+tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
res=pd.DataFrame([metrics(yev,pB,.5,'baseline_rf_0.50'),metrics(yev,pC,.5,'coral_rf_0.50'),metrics(yev,pC,T,'coral_rf_source_threshold')]); display(res); res.to_csv(TABLE/'03_frozen_cicids_results.csv',index=False)

## 9. Target threshold sensitivity — diagnostic only

In [ ]:
diag=pd.DataFrame([metrics(yev,pC,t,'diagnostic') for t in grid]); diag.to_csv(TABLE/'04_target_threshold_diagnostic.csv',index=False)
plt.figure(figsize=(9,5)); plt.plot(diag.threshold,diag.f1); plt.axvline(T,linestyle='--',label=f'source threshold={T:.2f}'); plt.xlabel('Threshold'); plt.ylabel('CICIDS F1'); plt.title('Target sensitivity (diagnostic only)'); plt.legend(); plt.tight_layout(); plt.savefig(PLOTS/'02_target_threshold_sensitivity.png',dpi=160); plt.show()

## 10. Attack-family effect at the frozen threshold

In [ ]:
labels=cicids.Label.astype(str).str.strip().iloc[ceval].reset_index(drop=True); bpred=(pB>=.5).astype('int8'); cpred=(pC>=T).astype('int8'); rows=[]
for lab in sorted(labels.unique()):
    m=labels.eq(lab).to_numpy(); rows.append({'attack_label':lab,'n':int(m.sum()),'baseline_detection_rate':bpred[m].mean(),'coral_frozen_detection_rate':cpred[m].mean(),'change':cpred[m].mean()-bpred[m].mean()})
att=pd.DataFrame(rows).sort_values(['n','change'],ascending=[False,False]); display(att); att.to_csv(TABLE/'05_attack_family_effect.csv',index=False)

## 11. Research decision gate

- **If frozen-threshold CORAL improves transfer:** evidence supports threshold/calibration shift as an important mechanism.
- **If F1 improves but FPR becomes unacceptable:** report the operating-point trade-off; do not call it a complete solution.
- **If threshold transfer still fails:** threshold alone does not explain the gap. The next targeted experiment should investigate class-prior shift and attack-family-specific representation mismatch.

**Important:** the target sensitivity curve is diagnostic only and must never be used to select the reported threshold.

In [ ]:
evidence={'frozen_source_threshold':T,'covariance_distance_before':float(cb),'covariance_distance_after':float(ca),'results':res.to_dict(orient='records')}
with open(TABLE/'06_phase11_evidence.json','w') as f: json.dump(evidence,f,indent=2)
print('Phase 11 outputs saved to',OUT)